# TestCompass | GA4 Exploratory Data Analysis

## Objective
Assess data quality and explore patterns associated with purchasing after a session’s first product view. Findings will inform purchase prediction and generate hypotheses for future experiments.

## Dataset
Google’s public GA4 sample e-commerce data, exported from BigQuery.

- Period: 1 November 2020 to 31 January 2021
- Expected size: 76,995 sessions and 14 columns
- Each row represents an eligible session with a product view
- Target: `purchased_later`, indicating a purchase after the first product view within the same session

## Scope and limitations
Predictors use information available at or before the first product view. Sessions with a purchase before or at that point are excluded.

The data is obfuscated. Observed patterns show associations and cannot establish which experiment will cause an improvement.

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_csv(
    root / "data/raw/ga4/ga4_session_features.csv",
    dtype={"user_pseudo_id": "string", "session_id": "string"}
)

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

Rows: 76,995 | Columns: 14


,user_pseudo_id,session_id,prediction_date,prediction_timestamp,device_category,first_user_source,first_user_medium,seconds_before_first_view,prior_events,prior_page_views,prior_search_events,prior_cart_events,prior_first_visit_events,purchased_later
0,3380520.8884755581,6568989672,2020-11-01,2020-11-01 00:07:50.895046 UTC,desktop,(direct),(none),80.658108,4,1,0,0,1,0
1,36638696.4029295753,6944996097,2020-11-01,2020-11-01 00:10:35.593253 UTC,mobile,<Other>,<Other>,40.475932,7,3,0,0,1,1
2,88098550.2597192571,3203951494,2020-11-01,2020-11-01 00:26:40.498688 UTC,desktop,shop.googlemerchandisestore.com,referral,42.122458,6,2,0,0,1,0
3,5551476.3360070688,4338623168,2020-11-01,2020-11-01 00:29:05.399028 UTC,desktop,(direct),(none),203.020026,11,3,0,0,1,0
4,3015192.8617504010,6300366416,2020-11-01,2020-11-01 00:33:40.881619 UTC,desktop,<Other>,<Other>,369.448321,71,33,1,0,1,0


## 1. Load the dataset
Load the GA4 session export and verify its dimensions and column structure.

In [2]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_csv(
    root / "data/raw/ga4/ga4_session_features.csv",
    dtype={"user_pseudo_id": "string", "session_id": "string"}
)

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

Rows: 76,995 | Columns: 14


,user_pseudo_id,session_id,prediction_date,prediction_timestamp,device_category,first_user_source,first_user_medium,seconds_before_first_view,prior_events,prior_page_views,prior_search_events,prior_cart_events,prior_first_visit_events,purchased_later
0,3380520.8884755581,6568989672,2020-11-01,2020-11-01 00:07:50.895046 UTC,desktop,(direct),(none),80.658108,4,1,0,0,1,0
1,36638696.4029295753,6944996097,2020-11-01,2020-11-01 00:10:35.593253 UTC,mobile,<Other>,<Other>,40.475932,7,3,0,0,1,1
2,88098550.2597192571,3203951494,2020-11-01,2020-11-01 00:26:40.498688 UTC,desktop,shop.googlemerchandisestore.com,referral,42.122458,6,2,0,0,1,0
3,5551476.3360070688,4338623168,2020-11-01,2020-11-01 00:29:05.399028 UTC,desktop,(direct),(none),203.020026,11,3,0,0,1,0
4,3015192.8617504010,6300366416,2020-11-01,2020-11-01 00:33:40.881619 UTC,desktop,<Other>,<Other>,369.448321,71,33,1,0,1,0


## 2. Check data quality
Inspect data types, missing values and duplicate session keys before analysis.

In [3]:
df.info()

display(
    df.isna().sum()
      .rename("missing_values")
      .to_frame()
)

duplicate_sessions = df.duplicated(
    subset=["user_pseudo_id", "session_id"]
).sum()

print(f"Duplicate session keys: {duplicate_sessions:,}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76995 entries, 0 to 76994
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   user_pseudo_id             76995 non-null  string 
 1   session_id                 76995 non-null  string 
 2   prediction_date            76995 non-null  object 
 3   prediction_timestamp       76995 non-null  object 
 4   device_category            76995 non-null  object 
 5   first_user_source          76995 non-null  object 
 6   first_user_medium          76995 non-null  object 
 7   seconds_before_first_view  76995 non-null  float64
 8   prior_events               76995 non-null  int64  
 9   prior_page_views           76995 non-null  int64  
 10  prior_search_events        76995 non-null  int64  
 11  prior_cart_events          76995 non-null  int64  
 12  prior_first_visit_events   76995 non-null  int64  
 13  purchased_later            76995 non-null  int

,missing_values
user_pseudo_id,0
session_id,0
prediction_date,0
prediction_timestamp,0
device_category,0
first_user_source,0
first_user_medium,0
seconds_before_first_view,0
prior_events,0
prior_page_views,0


Duplicate session keys: 0


### Finding
- No missing values or duplicate visitor-session keys were detected. Further checks are needed for extreme values and feature consistency.

## 3. Inspect the target
Check the balance between sessions with and without a subsequent purchase.

In [4]:
target_summary = (
    df["purchased_later"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("sessions")
    .to_frame()
)

target_summary["percentage"] = (
    target_summary["sessions"] / len(df) * 100
).round(2)

display(target_summary)

,sessions,percentage
purchased_later,,
0,72322,93.93
1,4673,6.07


### Finding
- Only 6.07% of eligible sessions resulted in a subsequent purchase. A model predicting no purchases would achieve 93.93% accuracy, so accuracy alone is insufficient.

## 4. Explore purchase rates over time
Compare monthly session volumes and purchase rates to identify changes relevant to a chronological model evaluation.

In [5]:
df["prediction_date"] = pd.to_datetime(
    df["prediction_date"].astype(str),
    format="%Y-%m-%d",
    errors="raise"
)

monthly_summary = (
    df.groupby(df["prediction_date"].dt.to_period("M"))
    .agg(
        sessions=("purchased_later", "size"),
        purchasing_sessions=("purchased_later", "sum"),
        purchase_rate=("purchased_later", "mean")
    )
)

monthly_summary["purchase_rate_pct"] = (
    monthly_summary["purchase_rate"] * 100
).round(2)

monthly_summary = monthly_summary.drop(columns="purchase_rate")

display(monthly_summary)

,sessions,purchasing_sessions,purchase_rate_pct
prediction_date,,,
2020-11,25934,1580,6.09
2020-12,27962,2023,7.23
2021-01,23099,1070,4.63


### Finding
- Purchase rates varied from 6.09% in November to 7.23% in December and 4.63% in January. 
- Model evaluation should use later dates to assess performance under changing conditions.

## 5. Compare purchase rates by device
Compare session volumes and subsequent purchase rates across device categories. Differences may suggest experiment hypotheses but do not establish causation.

In [6]:
device_summary = (
    df.groupby("device_category", dropna=False)
    .agg(
        sessions=("purchased_later", "size"),
        purchasing_sessions=("purchased_later", "sum"),
        purchase_rate=("purchased_later", "mean")
    )
    .sort_values("sessions", ascending=False)
)

device_summary["purchase_rate_pct"] = (
    device_summary["purchase_rate"] * 100
).round(2)

device_summary = device_summary.drop(columns="purchase_rate")

display(device_summary)

,sessions,purchasing_sessions,purchase_rate_pct
device_category,,,
desktop,44807,2658,5.93
mobile,30488,1915,6.28
tablet,1700,100,5.88


### Finding
- Mobile had a higher observed purchase rate among eligible sessions. This descriptive difference requires further analysis before informing experiment priorities.

## 6. Explore first-user acquisition
Compare session volumes and purchase rates by first-user source and medium. These fields describe initial acquisition, not attribution for the current session.

In [7]:
acquisition_summary = (
    df.groupby(
        ["first_user_source", "first_user_medium"],
        dropna=False
    )
    .agg(
        sessions=("purchased_later", "size"),
        purchasing_sessions=("purchased_later", "sum"),
        purchase_rate=("purchased_later", "mean")
    )
    .sort_values("sessions", ascending=False)
)

acquisition_summary["purchase_rate_pct"] = (
    acquisition_summary["purchase_rate"] * 100
).round(2)

acquisition_summary = acquisition_summary.drop(columns="purchase_rate")

display(acquisition_summary)

sessions  \
first_user_source               first_user_medium             
google                          organic               23660   
(direct)                        (none)                17671   
<Other>                         <Other>               10588   
                                referral               7454   
shop.googlemerchandisestore.com referral               6629   
(data deleted)                  (data deleted)         5648   
google                          cpc                    3167   
<Other>                         organic                2077   
                                (data deleted)           99   
(data deleted)                  referral                  1   
<Other>                         cpc                       1   

                                                   purchasing_sessions  \
first_user_source               first_user_medium                        
google                          organic                           1206   
(direct)                        (none)                            1049   
<Other>                         <Other>                            492   
                                referral                           454   
shop.googlemerchandisestore.com referral                           558   
(data deleted)                  (data deleted)                     661   
google                          cpc                                150   
<Other>                         organic                             95   
                                (data deleted)                       8   
(data deleted)                  referral                             0   
<Other>                         cpc                                  0   

                                                   purchase_rate_pct  
first_user_source               first_user_medium                     
google                          organic                         5.10  
(direct)                        (none)                          5.94  
<Other>                         <Other>                         4.65  
                                referral                        6.09  
shop.googlemerchandisestore.com referral                        8.42  
(data deleted)                  (data deleted)                 11.70  
google                          cpc                             4.74  
<Other>                         organic                         4.57  
                                (data deleted)                  8.08  
(data deleted)                  referral                        0.00  
<Other>                         cpc                             0.00

### Findings
- Google organic had the largest eligible session volume, with a 5.10% purchase rate. Direct traffic converted at 5.94% and Google CPC at 4.74%.

- The highest rate, 11.70%, belonged to `(data deleted)`, which cannot identify an actionable channel. Obscured sources and referrals from the store’s own domain limit interpretation.

- These fields describe first-user acquisition, not current-session attribution. The differences do not establish channel effectiveness.

## 7. Inspect numerical features
Examine distributions and extreme values before deciding whether cleaning or transformations are needed.

In [8]:
numeric_features = [
    "seconds_before_first_view",
    "prior_events",
    "prior_page_views",
    "prior_search_events",
    "prior_cart_events",
    "prior_first_visit_events"
]

display(
    df[numeric_features]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .T
    .round(2)
)

,count,mean,std,min,50%,90%,95%,99%,max
seconds_before_first_view,76995.0,134.43,1482.67,0.0,38.62,309.34,544.84,1576.76,402477.09
prior_events,76995.0,8.30,10.53,0.0,5.00,15.60,21.00,81.00,244.00
prior_page_views,76995.0,3.06,4.71,0.0,2.00,6.00,8.00,38.00,162.00
prior_search_events,76995.0,0.07,0.35,0.0,0.00,0.00,1.00,1.00,21.00
prior_cart_events,76995.0,0.00,0.04,0.0,0.00,0.00,0.00,0.00,3.00
prior_first_visit_events,76995.0,0.66,0.47,0.0,1.00,1.00,1.00,1.00,2.00


### Findings
- Time before the first product view is strongly right-skewed: the median is 38.62 seconds, compared with a mean of 134.43 seconds. The maximum is approximately 4.66 days and needs investigation before modeling.

- Prior event and page-view counts also have long upper tails. Prior cart events are almost always zero, so their usefulness needs assessment.

- Some sessions contain two prior first-visit events, which requires a consistency check. Extreme values will be investigated before deciding whether to retain, transform or exclude them.

## 8. Investigate unusual feature values

Quantify sessions with unusually long times before the first product view,
cart events before that view and repeated first-visit events.

These checks flag records for investigation

In [9]:
checks = {
    "Time before first view > 30 minutes":
        df["seconds_before_first_view"] > 1800,
    "Time before first view > 1 hour":
        df["seconds_before_first_view"] > 3600,
    "Time before first view > 1 day":
        df["seconds_before_first_view"] > 86400,
    "Cart events before first product view":
        df["prior_cart_events"] > 0,
    "More than one prior first-visit event":
        df["prior_first_visit_events"] > 1
}

check_summary = pd.DataFrame([
    {
        "check": name,
        "sessions": int(mask.sum()),
        "percentage_of_sessions": round(mask.mean() * 100, 3)
    }
    for name, mask in checks.items()
])

display(check_summary)

print("Ten sessions with the longest time before first product view:")

display(
    df.nlargest(10, "seconds_before_first_view")[
        [
            "user_pseudo_id",
            "session_id",
            "prediction_date",
            "seconds_before_first_view",
            "prior_events",
            "prior_page_views",
            "prior_cart_events",
            "prior_first_visit_events",
            "purchased_later"
        ]
    ]
)

,check,sessions,percentage_of_sessions
0,Time before first view > 30 minutes,496,0.644
1,Time before first view > 1 hour,49,0.064
2,Time before first view > 1 day,1,0.001
3,Cart events before first product view,27,0.035
4,More than one prior first-visit event,8,0.010


Ten sessions with the longest time before first product view:


,user_pseudo_id,session_id,prediction_date,seconds_before_first_view,prior_events,prior_page_views,prior_cart_events,prior_first_visit_events,purchased_later
5215,69820731.5577222848,5029350990,2020-11-07,402477.090792,2,1,0,0,0
23597,6211525.7294516618,4774120987,2020-11-28,15028.417649,16,4,0,1,0
40864,8147063.1474060130,6216045559,2020-12-14,14322.758909,3,1,0,1,0
46897,16144971.6490588498,8564675088,2020-12-19,8101.092593,107,47,0,0,0
46216,1835954.3661646834,9707262704,2020-12-18,7226.221284,27,11,0,1,0
46112,53846269.9706707177,1850527124,2020-12-18,6966.384282,33,14,0,1,0
70044,79640807.0397697073,1082869209,2021-01-23,6017.499995,35,13,0,0,0
76229,1973757.8722046630,930411518,2021-01-30,5791.461319,110,52,0,0,0
69004,27865160.2048337374,76092773,2021-01-22,5698.712344,24,9,0,0,0
39348,57157947.6829094748,9977983438,2020-12-12,5550.616713,92,38,0,0,0


### Findings & decision

- Additional checks in BigQuery identified 22 sessions (0.029%)
with gaps above 30 minutes between observed events before or
at the first product view. Six had gaps above one hour.
The largest gap was 4.66 days.

- All 22 flagged sessions had no subsequent recorded purchase.
The cause remains unresolved. Records are retained and the
original CSV is unchanged.

- This dataset contains only eligible product-view sessions.
Its 6.07% purchase rate does not represent all website sessions.
A separate export covering all sessions is needed for the
broader website audit dashboard

## 9. Validate the initial dashboard session export
Load the all-session dataset separately from the prediction dataset.
Check export completeness, date coverage and duplicate session keys.

In [11]:
dashboard_df = pd.read_csv(
    root / "data/raw/ga4/ga4_dashboard_sessions.csv",
    dtype={"user_pseudo_id": "string", "session_id": "string"},
    parse_dates=["session_date"]
)

print(f"Rows: {len(dashboard_df):,} | Columns: {dashboard_df.shape[1]}")
print(
    "Period:",
    dashboard_df["session_date"].min().date(),
    "to",
    dashboard_df["session_date"].max().date()
)
print("Unique dates:", dashboard_df["session_date"].nunique())
print(
    "Duplicate session keys:",
    dashboard_df.duplicated(["user_pseudo_id", "session_id"]).sum()
)

display(dashboard_df.isna().sum().rename("missing_values").to_frame())

Rows: 360,129 | Columns: 23
Period: 2020-11-01 to 2021-01-31
Unique dates: 92
Duplicate session keys: 0


,missing_values
user_pseudo_id,0
session_id,0
session_date,0
first_observed_time,0
last_observed_time,0
device_category,0
first_user_source,0
first_user_medium,0
event_count,0
page_views,0


## 10. Audit purchase revenue and transaction IDs

This audit uses the raw GA4 purchase events for 1 November 2020 to 31 January 2021. It is separate from the eligible-session prediction dataset and the all-session dashboard export.

### Results

- The dataset contains 5,692 purchase-event rows. All 5,692 have a non-null `purchase_revenue_in_usd` value.
- 5,242 rows have positive revenue; 450 have a value of zero.
- The sum across purchase-event rows is $362,165 USD, ranging from $0 to $1,530.
- Transaction IDs are not reliable for counting or deduplicating orders. The placeholder `(not set)` appears on 883 purchase events, and some repeated IDs have different revenue values.

### Interpretation and limitations

The $362,165 figure is an event-level sum from an obfuscated sample. It is not validated, transaction-deduplicated revenue. The dashboard session export includes `purchase_events` and a `purchased` flag, but does not include purchase revenue.

Therefore, this project reports purchase activity, session conversion and funnel progression. It does not report verified order revenue or revenue impact. Purchase revenue is also excluded from the prediction features because it occurs after the purchase outcome.

Source: [Google Analytics BigQuery sample dataset and limitations](https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset)

## 11. Validate the final dashboard export

The initial dashboard export above contains 23 columns.
The final version adds ordered-funnel fields and audit flags,
giving 35 columns for the same all-session population.

This validation checks the final session CSV used in Tableau.

In [14]:
final_dashboard_df = pd.read_csv(
    root / "data/raw/ga4/ga4_dashboard_sessions_v2.csv",
    dtype={
        "user_pseudo_id": "string",
        "session_id": "string"
    },
    parse_dates=["session_date"]
)

print(
    f"Rows: {len(final_dashboard_df):,} | "
    f"Columns: {final_dashboard_df.shape[1]}"
)
print(
    "Period:",
    final_dashboard_df["session_date"].min().date(),
    "to",
    final_dashboard_df["session_date"].max().date()
)
print("Unique dates:", final_dashboard_df["session_date"].nunique())
print(
    "Duplicate session keys:",
    final_dashboard_df.duplicated(
        ["user_pseudo_id", "session_id"]
    ).sum()
)

/var/folders/bb/jd9v798j0t96bmp2q4bsrn9h0000gn/T/ipykernel_19171/1266172908.py:1: DtypeWarning: Columns (24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  final_dashboard_df = pd.read_csv(


Rows: 360,129 | Columns: 35
Period: 2020-11-01 to 2021-01-31
Unique dates: 92
Duplicate session keys: 0


## 12. Analysis scope and statistical limitations

Sections 1–8 document earlier exploration of a product-view-only
dataset for a potential prediction task. That population differs
from the final all-session dashboard. No predictive model is
presented as a final deliverable.

The final dashboard describes website performance and recorded
funnel progression. The opportunity ranking combines session volume
with a progression-rate shortfall against a comparison group.

This ranking prioritises investigation. It does not establish
statistical significance, causation or expected experiment uplift.

No randomised experiment was conducted. Streamlit demonstrates a
workflow for developing, planning and documenting future website
A/B tests. Inferential analysis remains a possible extension.